### Formula 1 Object detection model (YOLOv26)

#### Environment

`(1) Packages` 

In [ ]:
%pip install ultralytics
%pip install fiftyone

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from ultralytics import YOLO
import cv2
import pandas as pd
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import subprocess

#### Part 1: Collect Data

In [8]:
video_folder = Path("D:/new_pc/engineershit/github_projects/Computer_Vision/datasets/videos_input")
output_folder = Path("D:/new_pc/engineershit/github_projects/Computer_Vision/datasets/output_output")

for video in video_folder.glob("*.mp4"):


    race_output = output_folder / video.stem
    race_output.mkdir(parents=True, exist_ok=True)

    output_pattern = race_output / "frame_%05d.jpg"

    subprocess.run([
        "ffmpeg",
        "-i", str(video),
        "-vf", "fps=1",
        str(output_pattern)
    ])

#### Part 2: Prepare model environment

In [ ]:
#Data
F1_CV_project_folder = Path("D:/new_pc/engineershit/github_projects/Computer_Vision/F1_CV_project")

#Save
run_name_1 = "F1_CV_1"

#### Part 3: Create model

In [ ]:
model = YOLO("yolo26n.pt")

train = model.train(
    data = "D:/new_pc/engineershit/github_projects/Computer_Vision/datasets/approved_dataset/data.yaml",
    epochs = 100,
    imgsz = 640,
    project = F1_CV_project_folder,
    name = run_name_1,
    exist_ok = False
)

Ultralytics 8.4.127  Python-3.14.6 torch-2.13.0+cpu CPU (Intel Core i5-10400F 2.90GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:/new_pc/engineershit/github_projects/Computer_Vision/datasets/approved_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, m

#### Part 4: Prepare evaluation environment

In [ ]:
video_folder = Path("D:/new_pc/engineershit/github_projects/Computer_Vision/datasets/videos_input")
output_csv = Path("D:/new_pc/engineershit/github_projects/Computer_Vision/datasets/videos_output/output.csv")
output_video_folder = Path("D:/new_pc/engineershit/github_projects/Computer_Vision/datasets/videos_output")

#### Part 5: Create visual evaluation

`(1) Packages` 

In [ ]:
rows = []

`(2) Track Cars and collect Coordiantes` 


In [ ]:
cap = cv2.VideoCapture(video_path)


frame_idx = 0


while cap.isOpened():


    success, frame = cap.read()


    if not success:
        break


    results = trained_model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.25,
        iou=0.5,
        verbose=False
    )


    result = results[0]


    if result.boxes is not None and result.boxes.id is not None:


        boxes = result.boxes


        xyxy = boxes.xyxy.cpu().numpy()


        track_ids = boxes.id.cpu().numpy()


        confidences = boxes.conf.cpu().numpy()


        class_ids = boxes.cls.cpu().numpy()


        for box, track_id, confidence, class_id in zip(
            xyxy, track_ids, confidences, class_ids
        ):


            x1, y1, x2, y2 = box


            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2


            width = x2 - x1
            height = y2 - y1


            time_seconds = frame_idx / fps


            rows.append({
                "video_path": video_path,
                "frame": frame_idx,
                "time_seconds": time_seconds,
                "track_id": int(track_id),
                "class_id": int(class_id),
                "confidence": float(confidence),
                "x1": float(x1),
                "y1": float(y1),
                "x2": float(x2),
                "y2": float(y2),
                "cx": float(cx),
                "cy": float(cy),
                "width": float(width),
                "height": float(height),
                "cx_norm": float(cx / frame_width),
                "cy_norm": float(cy / frame_height)
            })


    frame_idx += 1


    if frame_idx % 100 == 0:
        print(f"Processed frame {frame_idx}")


cap.release()

print("Tracking finished.")
print("Number of collected detections:", len(rows))

`(3) Save data to .csv` 

In [ ]:
df = pd.DataFrame(rows)


if df.empty:
    print("No tracking data was collected.")
    print("Try checking the model, video path, or lowering conf from 0.25 to 0.15.")
else:

    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)


    df.to_csv(output_csv, index=False)

    print("Saved trajectory CSV to:", output_csv)
    print("Number of rows:", len(df))
    print(df.head())

`(4) Open Video` 

In [ ]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError("Video could not be opened. Check video_path.")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Video opened successfully.")
print("FPS:", fps)
print("Frame width:", frame_width)
print("Frame height:", frame_height)
print("Total frames:", total_frames)


`(5) Create video writer` 

In [ ]:
Path(output_video_path).parent.mkdir(parents=True, exist_ok=True)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_video_path,
    fourcc,
    fps,
    (frame_width, frame_height)
)